# Community Analysis, Key Bridges, and Liaison Discovery

## Setup

The notebook first loads the clean allocation data, rebuilds the yearly repo-sharing communities if needed, and then walks through three analysis blocks:

1. Community structure
2. Key bridges
3. Liaison discovery


In [ ]:
import sys, warnings; sys.path.append('..')
warnings.filterwarnings('ignore')

import pickle
import random
from pathlib import Path

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedFormatter, FixedLocator, NullFormatter
from networkx.algorithms.community import louvain_communities, modularity

from nersc_graphs import generate_graph
from pipeline import extract, io as cio

PALETTE = ['#2F6DA3', '#F28E2B', '#D1495B', '#2A9D8F', '#6C757D', '#7A6FAC']
sns.set_theme(style='whitegrid')
sns.set_palette(PALETTE)
plt.rcParams.update({
    'axes.prop_cycle': plt.cycler(color=PALETTE),
    'figure.dpi': 120,
    'axes.titlesize': 18,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
})

YEARS = extract.YEARS
FULL_YEARS = [y for y in YEARS if y < max(YEARS)]
BRIDGE_ANALYSIS_YEARS = FULL_YEARS
PREVIEW_YEAR = max(YEARS)
DISPLAY_YEAR = max(FULL_YEARS)
LIAISON_YEAR = DISPLAY_YEAR

NULL_TRIALS = 100
NULL_SWAP_MULTIPLIER = 10
NULL_SEED = 42
BRIDGE_TOP_DECILE = 0.90
BRIDGE_MIN_YEARS = 5
BRIDGE_MIN_OFFICES = 2
BRIDGE_BETWEENNESS_K_OBSERVED = None
BRIDGE_BETWEENNESS_K_NULL = 500
RANDOM_REMOVAL_TRIALS = 200


In [ ]:
alloc = cio.read_all_allocations()
repo_features = cio.read_alloc_repo_year_features()
user_features = cio.read_alloc_user_year_features()

def mode_or_none(s):
    m = s.mode(dropna=True)
    return m.iloc[0] if len(m) else None

def shannon_entropy(values):
    vals = pd.Series(values).dropna()
    if vals.empty:
        return np.nan
    probs = vals.value_counts(normalize=True)
    return float(-(probs * np.log2(probs)).sum())

def degree_preserving_null(G, seed=42, nswap_multiplier=10):
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from((u, v) for u, v in G.edges())
    if H.number_of_edges() >= 2:
        nswap = max(H.number_of_edges() * nswap_multiplier, 1)
        try:
            nx.double_edge_swap(H, nswap=nswap, max_tries=nswap * 20, seed=seed)
        except Exception:
            pass
    weights = [data.get('weight', 1.0) for _, _, data in G.edges(data=True)]
    rng = random.Random(seed)
    rng.shuffle(weights)
    for (u, v), w in zip(H.edges(), weights):
        H[u][v]['weight'] = w
    return H

def bridge_year_counts_from_graph(G, office_map, seed=42, k=None):
    if G.number_of_nodes() == 0:
        return pd.DataFrame(columns=['repo', 'betweenness', 'neighbor_offices', 'is_bridge_year'])
    if k is None:
        btw = nx.betweenness_centrality(G, weight='weight')
    else:
        k_eff = min(k, G.number_of_nodes())
        btw = nx.betweenness_centrality(G, weight='weight', k=k_eff, seed=seed)
    rows = []
    for repo in G.nodes():
        nbr_offices = {office_map.get(n) for n in G.neighbors(repo) if office_map.get(n)}
        rows.append({
            'repo': repo,
            'betweenness': float(btw.get(repo, 0.0)),
            'neighbor_offices': len(nbr_offices),
        })
    out = pd.DataFrame(rows)
    thresh = out['betweenness'].quantile(BRIDGE_TOP_DECILE)
    out['is_bridge_year'] = (
        (out['betweenness'] >= thresh) &
        (out['neighbor_offices'] >= BRIDGE_MIN_OFFICES)
    )
    return out

def cross_office_edge_count(G, office_map):
    count = 0
    for u, v in G.edges():
        ou = office_map.get(u)
        ov = office_map.get(v)
        if ou and ov and ou != ov:
            count += 1
    return int(count)

CACHE_PATH = Path('../clean/04_communities_cache.pkl')
if CACHE_PATH.exists():
    cache = pickle.loads(CACHE_PATH.read_bytes())
    communities = cache['communities']
    community_summary = cache['community_summary']
    mod_df = cache['mod_df']
    null_bridge_df = cache['null_bridge_df']
    graph_by_year = cache['graph_by_year']
    office_map_by_year = cache['office_map_by_year']
    science_map_by_year = cache['science_map_by_year']
    print(f'loaded cache from {CACHE_PATH}')
else:
    graph_by_year = {}
    office_map_by_year = {}
    science_map_by_year = {}
    community_rows = []
    community_summary_rows = []
    modularity_rows = []
    null_bridge_rows = []

    for y in YEARS:
        df_y = alloc[alloc['year'] == y]
        G, _ = generate_graph(df_y, node_name='repo', edge_name='user_id')
        graph_by_year[y] = G
        office_map = df_y.groupby('repo')['office'].agg(mode_or_none).to_dict()
        science_map = df_y.groupby('repo')['science_category'].agg(mode_or_none).to_dict()
        office_map_by_year[y] = office_map
        science_map_by_year[y] = science_map
        if G.number_of_nodes() == 0:
            continue

        parts = louvain_communities(G, weight='weight', seed=NULL_SEED)
        obs_mod = modularity(G, parts, weight='weight')
        null_mods = []
        for t in range(NULL_TRIALS):
            H = degree_preserving_null(
                G,
                seed=NULL_SEED + y * 1000 + t,
                nswap_multiplier=NULL_SWAP_MULTIPLIER,
            )
            null_parts = louvain_communities(H, weight='weight', seed=NULL_SEED)
            null_mods.append(modularity(H, null_parts, weight='weight'))
            null_bridge = bridge_year_counts_from_graph(
                H,
                office_map,
                seed=NULL_SEED + t,
                k=BRIDGE_BETWEENNESS_K_NULL,
            )
            null_bridge_rows.append({
                'year': y,
                'trial': t,
                'n_bridge_year_repos': int(null_bridge['is_bridge_year'].sum()),
            })

        null_mean = float(np.mean(null_mods))
        null_std = float(np.std(null_mods, ddof=1)) if len(null_mods) > 1 else np.nan
        zscore = float((obs_mod - null_mean) / null_std) if null_std and null_std > 0 else np.nan
        modularity_rows.append({
            'year': y,
            'modularity': obs_mod,
            'n_communities': len(parts),
            'null_mean': null_mean,
            'null_std': null_std,
            'null_lo': float(np.percentile(null_mods, 2.5)),
            'null_hi': float(np.percentile(null_mods, 97.5)),
            'modularity_z': zscore,
            'n_nodes': G.number_of_nodes(),
            'n_edges': G.number_of_edges(),
        })

        repo_hours = (
            df_y.groupby('repo')
            .agg(
                cpu_h=('cpu_node_hours_charged', 'sum'),
                gpu_h=('gpu_node_hours_charged', 'sum'),
            )
        )
        denom = repo_hours['cpu_h'] + repo_hours['gpu_h']
        repo_gpu = (repo_hours['gpu_h'] / denom.where(denom > 0)).to_dict()
        repo_cpu_h_map = repo_hours['cpu_h'].to_dict()
        repo_gpu_h_map = repo_hours['gpu_h'].to_dict()

        for cid, comm in enumerate(parts):
            repos = sorted(comm)
            offices = [office_map.get(r) for r in repos]
            sciences = [science_map.get(r) for r in repos]
            mean_gpu = float(np.nanmean([repo_gpu.get(r, np.nan) for r in repos])) if repos else np.nan
            total_cpu = sum(repo_cpu_h_map.get(r, 0.0) for r in repos)
            total_gpu = sum(repo_gpu_h_map.get(r, 0.0) for r in repos)
            weighted_gpu = (total_gpu / (total_cpu + total_gpu)) if (total_cpu + total_gpu) > 0 else np.nan
            community_summary_rows.append({
                'year': y,
                'community_id': cid,
                'n_repos': len(repos),
                'dominant_office': mode_or_none(pd.Series(offices)),
                'dominant_science': mode_or_none(pd.Series(sciences)),
                'office_entropy': shannon_entropy(offices),
                'mean_repo_gpu_share': mean_gpu,
                'hours_weighted_gpu_share': weighted_gpu,
            })
            for repo in repos:
                community_rows.append({
                    'repo': repo,
                    'year': y,
                    'community_id': cid,
                    'office': office_map.get(repo),
                    'science_category': science_map.get(repo),
                    'repo_gpu_share': repo_gpu.get(repo, np.nan),
                })

    communities = pd.DataFrame(community_rows)
    community_summary = pd.DataFrame(community_summary_rows)
    mod_df = pd.DataFrame(modularity_rows)
    null_bridge_df = pd.DataFrame(null_bridge_rows)

    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_bytes(pickle.dumps({
        'communities': communities,
        'community_summary': community_summary,
        'mod_df': mod_df,
        'null_bridge_df': null_bridge_df,
        'graph_by_year': graph_by_year,
        'office_map_by_year': office_map_by_year,
        'science_map_by_year': science_map_by_year,
    }))
    print(f'saved cache to {CACHE_PATH}')

print('years:', YEARS)


## 1. Community Structure

This block asks whether the repo-sharing graph has meaningful communities, how mixed those communities are across funding offices, how stable they remain over time, and how GPU usage varies across them.

### 1.1 Modularity vs degree-preserving null

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 6))
ax1.axvspan(PREVIEW_YEAR - 0.5, PREVIEW_YEAR + 0.5, color='lightgrey', alpha=0.35,
            label=f'{PREVIEW_YEAR} preview (partial year)')
ax1.fill_between(mod_df['year'], mod_df['null_lo'], mod_df['null_hi'],
                 color='tab:blue', alpha=0.16, label='degree-preserving null (95% band)')
sns.lineplot(data=mod_df, x='year', y='null_mean', ax=ax1,
             color=PALETTE[0], linestyle='--', linewidth=2.8, label='null mean')
sns.lineplot(data=mod_df, x='year', y='modularity', ax=ax1,
             color=PALETTE[0], marker='o', linewidth=3.2, markersize=10,
             label='observed modularity')
pm = mod_df['year'] == PREVIEW_YEAR
ax1.plot(mod_df.loc[pm, 'year'], mod_df.loc[pm, 'modularity'],
         marker='o', markerfacecolor='white', markeredgecolor='tab:blue',
         markeredgewidth=2, linestyle='None', markersize=12,
         label=f'observed {PREVIEW_YEAR} preview')
ax1.set_ylabel('modularity')
ax1.set_xlabel('year')
ax1.set_title('Modularity vs degree-preserving null over time')
lines1, labels1 = ax1.get_legend_handles_labels()
ax1.legend(lines1, labels1, loc='upper right', bbox_to_anchor=(1.0, 0.74),
           frameon=True, framealpha=1.0)
plt.show()

mod_df[['year', 'modularity', 'null_mean', 'null_lo', 'null_hi', 'modularity_z', 'n_communities']]


### 1.2 Community size and office mixing across years

In [ ]:
landscape = (community_summary[community_summary['year'].isin(FULL_YEARS)]
             .query('n_repos >= 3').copy())
landscape['size_rank'] = (
    landscape.groupby('year')['n_repos']
    .rank(method='first', ascending=False).astype(int)
)
landscape['short_label'] = 'C' + landscape['community_id'].astype(int).astype(str)
landscape['entropy_band'] = pd.cut(
    landscape['office_entropy'].fillna(0),
    bins=[-0.01, 0.25, 1.0, 3.5],
    labels=['single-office', 'moderately mixed', 'highly mixed'],
)
plot_landscape = landscape[landscape['size_rank'] <= 12].copy()

fig, ax = plt.subplots(figsize=(12, 7))
sns.swarmplot(
    data=plot_landscape,
    x='year',
    y='n_repos',
    hue='entropy_band',
    hue_order=['single-office', 'moderately mixed', 'highly mixed'],
    palette=['#D1495B', '#F28E2B', '#2F6DA3'],
    size=10,
    alpha=0.9,
    edgecolor='white',
    linewidth=0.9,
    ax=ax,
)
year_pos = {y: i for i, y in enumerate(FULL_YEARS)}
for _, r in (
    plot_landscape.sort_values(['year', 'n_repos'], ascending=[True, False])
    .groupby('year').head(2).iterrows()
):
    ax.text(year_pos[r['year']] + 0.14, r['n_repos'] + 2.5, r['short_label'],
            fontsize=11, color='#333333')
ax.set_xlabel('year')
ax.set_ylabel('# projects in cluster')
ax.set_title('Community landscape: size and office mixing')
ax.legend(title='office mixing', loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
sns.despine(ax=ax)
plt.show()


### 1.3 Community-by-office composition

In [ ]:

c_full = communities[communities['year'] == DISPLAY_YEAR]
cross = c_full.pivot_table(index='community_id', columns='office',
                           values='repo', aggfunc='count', fill_value=0)
top_comms = cross.sum(axis=1).nlargest(15).index
heat = cross.loc[top_comms].copy()
column_order = heat.sum(axis=0).sort_values(ascending=False).index.tolist()
heat = heat[column_order]
heat.index = [f'C{cid}' for cid in heat.index]
office_short = {office: f'Office {i + 1}' for i, office in enumerate(heat.columns)}
heat = heat.rename(columns=office_short)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(heat, annot=True, fmt='d', cmap='Blues',
            cbar_kws={'label': '# projects'}, linewidths=0.4, linecolor='white',
            annot_kws={'size': 11}, ax=ax)
ax.set_title(f'Community x office ({DISPLAY_YEAR} top-15 communities)')
ax.set_xlabel('office')
ax.set_ylabel('community')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)
plt.show()


### 1.4 Are the biggest communities more cross-office than typical?

In [ ]:
TOP_COMPARE = 5
MIN_COMMUNITY_SIZE = 3

ec = (community_summary[community_summary['year'].isin(FULL_YEARS)]
      .dropna(subset=['office_entropy']).copy())
ec['size_rank'] = (
    ec.groupby('year')['n_repos']
    .rank(method='first', ascending=False).astype(int)
)

rows = []
for y in FULL_YEARS:
    sub = ec[ec['year'] == y]
    top = sub[sub['size_rank'] <= TOP_COMPARE]
    base = sub[sub['n_repos'] >= MIN_COMMUNITY_SIZE]
    rows.append({
        'year': y,
        'top_median_entropy': float(top['office_entropy'].median()),
        'all_median_entropy': float(base['office_entropy'].median()),
    })
year_compare = pd.DataFrame(rows)
plot_long = year_compare.melt(
    id_vars='year',
    value_vars=['top_median_entropy', 'all_median_entropy'],
    var_name='series',
    value_name='median_entropy',
)
plot_long['series'] = plot_long['series'].map({
    'top_median_entropy': f'top {TOP_COMPARE} median',
    'all_median_entropy': f'all communities size>={MIN_COMMUNITY_SIZE} median',
})

fig, ax = plt.subplots(figsize=(11, 6.5))
sns.lineplot(data=plot_long, x='year', y='median_entropy',
             hue='series', style='series', markers=True, dashes=False,
             linewidth=2.8, markersize=10,
             palette=[PALETTE[2], PALETTE[0]], ax=ax)
ax.fill_between(year_compare['year'],
                year_compare['all_median_entropy'],
                year_compare['top_median_entropy'],
                color=PALETTE[2], alpha=0.10)

label_kw = dict(
    fontsize=10,
    fontweight='bold',
    path_effects=[pe.withStroke(linewidth=3.0, foreground='white')],
)
for _, r in year_compare.iterrows():
    ax.text(r['year'], r['top_median_entropy'] + 0.12,
            f"{r['top_median_entropy']:.2f}",
            ha='center', va='bottom', color=PALETTE[2], **label_kw)
    ax.text(r['year'], r['all_median_entropy'] - 0.12,
            f"{r['all_median_entropy']:.2f}",
            ha='center', va='top', color=PALETTE[0], **label_kw)

ax.set_xticks(FULL_YEARS)
ax.set_xlabel('year')
ax.set_ylabel('median office entropy')
ax.set_title('Biggest communities vs typical communities')
lo_data = float(min(year_compare['all_median_entropy'].min(),
                    year_compare['top_median_entropy'].min()))
hi_data = float(max(year_compare['all_median_entropy'].max(),
                    year_compare['top_median_entropy'].max()))
ax.set_ylim(lo_data - 0.45, hi_data + 0.2)
ax.legend(title='', frameon=True, framealpha=1.0,
          loc='lower center', bbox_to_anchor=(0.5, 0.02),
          ncol=2, borderpad=0.6)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.show()


### 1.5 Persistence of the largest base-year communities

In [ ]:
def _jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(len(a | b), 1)

base_year = FULL_YEARS[0]
base = communities[communities['year'] == base_year].groupby('community_id')['repo'].apply(set)
rows = []
for y in FULL_YEARS:
    cur = communities[communities['year'] == y].groupby('community_id')['repo'].apply(set)
    for bid, bset in base.items():
        rows.append({
            'base_community': bid,
            'year': y,
            'best_jaccard': max((_jaccard(bset, cset) for cset in cur), default=0),
        })
persist = pd.DataFrame(rows)
top4_ids = base.map(len).sort_values(ascending=False).head(4).index.tolist()
persist_plot = persist[persist['base_community'].isin(top4_ids)].copy()
persist_plot['label'] = persist_plot['base_community'].map(
    lambda bid: f'C{int(bid)} (n={len(base[bid])})'
)

rng = np.random.default_rng(123)
N_NULL = 30
null_rows = []
for y in FULL_YEARS[1:]:
    year_df = communities[communities['year'] == y]
    for _ in range(N_NULL):
        sh = year_df.copy()
        sh['community_id'] = rng.permutation(sh['community_id'].values)
        sh_sets = sh.groupby('community_id')['repo'].apply(set)
        for bid in top4_ids:
            null_rows.append({
                'year': y,
                'base_community': bid,
                'best_jaccard': max((_jaccard(base[bid], cset) for cset in sh_sets), default=0),
            })
null_df = pd.DataFrame(null_rows)
null_band = (
    null_df.groupby('year')['best_jaccard']
    .agg(
        null_mean='mean',
        null_p95=lambda s: float(np.percentile(s, 95)),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 7))
sns.lineplot(data=persist_plot, x='year', y='best_jaccard',
             hue='label', style='label', markers=True, dashes=False,
             linewidth=3.0, markersize=10, ax=ax)
ax.fill_between(null_band['year'], 0, null_band['null_p95'],
                color=PALETTE[4], alpha=0.18, label='shuffled null 95% band')
ax.plot(null_band['year'], null_band['null_mean'],
        linestyle='--', color=PALETTE[4], linewidth=2.1, alpha=0.9,
        label='shuffled null mean')
ax.set_ylabel(f'best Jaccard vs {base_year}')
ax.set_xlabel('year')
ax.set_title(f'Persistence of the top {base_year} communities')
ax.set_ylim(0, 1.05)
ax.legend(title='', loc='upper right', frameon=False, ncol=2)
sns.despine(ax=ax)
plt.show()


### 1.6 Split and merge counts by transition year

In [ ]:
transition_rows = []
for source_year, dest_year in zip(FULL_YEARS[:-1], FULL_YEARS[1:]):
    src = communities[communities['year'] == source_year].groupby('community_id')['repo'].apply(set)
    dst = communities[communities['year'] == dest_year].groupby('community_id')['repo'].apply(set)
    for sid, sset in src.items():
        for did, dset in dst.items():
            overlap = len(sset & dset)
            if overlap == 0:
                continue
            transition_rows.append({
                'source_year': source_year,
                'dest_year': dest_year,
                'source_community': sid,
                'dest_community': did,
                'overlap': overlap,
                'source_size': len(sset),
                'dest_size': len(dset),
                'source_share': overlap / len(sset),
                'dest_share': overlap / len(dset),
            })
transitions = pd.DataFrame(transition_rows)

community_meta = community_summary.rename(columns={'n_repos': 'community_size'})

split_rows = []
for (year, cid), g in transitions.groupby(['source_year', 'source_community']):
    qual = g[(g['overlap'] >= 5) & (g['source_share'] >= 0.20)]
    if len(qual) >= 2:
        split_rows.append({
            'transition': f'{year}->{year + 1}',
            'source_year': year,
            'source_community': cid,
            'n_destinations': len(qual),
            'overlap_total': int(qual['overlap'].sum()),
        })
splits = pd.DataFrame(
    split_rows,
    columns=['transition', 'source_year', 'source_community', 'n_destinations', 'overlap_total'],
)

merge_rows = []
for (year, cid), g in transitions.groupby(['dest_year', 'dest_community']):
    qual = g[(g['overlap'] >= 5) & (g['dest_share'] >= 0.20)]
    if len(qual) >= 2:
        merge_rows.append({
            'transition': f'{year - 1}->{year}',
            'dest_year': year,
            'dest_community': cid,
            'n_sources': len(qual),
            'overlap_total': int(qual['overlap'].sum()),
        })
merges = pd.DataFrame(
    merge_rows,
    columns=['transition', 'dest_year', 'dest_community', 'n_sources', 'overlap_total'],
)

count_years = pd.DataFrame({'transition': [f'{y}->{y+1}' for y in FULL_YEARS[:-1]]})
count_years['splits'] = count_years['transition'].map(splits['transition'].value_counts()).fillna(0).astype(int)
count_years['merges'] = count_years['transition'].map(merges['transition'].value_counts()).fillna(0).astype(int)

count_long = count_years.melt(
    id_vars='transition',
    value_vars=['splits', 'merges'],
    var_name='event_type',
    value_name='n_events',
)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=count_long, x='transition', y='n_events', hue='event_type',
            palette=PALETTE[:2], ax=ax)
ax.set_ylabel('# events')
ax.set_xlabel('')
ax.set_title('Community split and merge counts by transition')
max_events = int(count_years[['splits', 'merges']].to_numpy().max())
ax.set_yticks(range(0, max_events + 1))
plt.show()

count_years


### 1.7 Which communities drive the biggest changes?

In [ ]:
split_detail = (
    splits.merge(
        community_meta,
        left_on=['source_year', 'source_community'],
        right_on=['year', 'community_id'],
        how='left',
    )
    .drop(columns=['year', 'community_id'])
    .sort_values(['overlap_total', 'n_destinations'], ascending=False)
)

merge_detail = (
    merges.merge(
        community_meta,
        left_on=['dest_year', 'dest_community'],
        right_on=['year', 'community_id'],
        how='left',
    )
    .drop(columns=['year', 'community_id'])
    .sort_values(['overlap_total', 'n_sources'], ascending=False)
)

split_view = split_detail[
    ['transition', 'source_community', 'dominant_office', 'dominant_science',
     'community_size', 'n_destinations', 'overlap_total']
].head(8).rename(columns={
    'source_community': 'community_id',
    'n_destinations': 'counterpart_count',
})
split_view['event_type'] = 'split'

merge_view = merge_detail[
    ['transition', 'dest_community', 'dominant_office', 'dominant_science',
     'community_size', 'n_sources', 'overlap_total']
].head(8).rename(columns={
    'dest_community': 'community_id',
    'n_sources': 'counterpart_count',
})
merge_view['event_type'] = 'merge'

top_change_events = pd.concat([split_view, merge_view], ignore_index=True)
top_change_events = top_change_events[
    ['event_type', 'transition', 'community_id', 'dominant_office',
     'dominant_science', 'community_size', 'counterpart_count', 'overlap_total']
]
top_change_events


### 1.8 GPU intensity vs office mixing

In [ ]:

OFFICE_PALETTE = PALETTE + ['#8DA750', '#B87333', '#9C755F', '#BAB0AC']

gpu_c = (community_summary[community_summary['year'] == DISPLAY_YEAR]
         .dropna(subset=['hours_weighted_gpu_share', 'mean_repo_gpu_share', 'dominant_office'])
         .query('n_repos >= 2').copy())
gpu_c['community_label'] = 'C' + gpu_c['community_id'].astype(int).astype(str)
offices_present = list(gpu_c['dominant_office'].dropna().unique())
all_offices = sorted(offices_present)
color_map = {o: OFFICE_PALETTE[i % len(OFFICE_PALETTE)] for i, o in enumerate(all_offices)}
office_short = {office: f'Office {i + 1}' for i, office in enumerate(all_offices)}

fig, ax = plt.subplots(figsize=(12, 7))
for _, r in gpu_c.iterrows():
    ax.plot([r['office_entropy']] * 2,
            [r['mean_repo_gpu_share'], r['hours_weighted_gpu_share']],
            color=color_map[r['dominant_office']], linewidth=1.4, alpha=0.6, zorder=1)
for office in all_offices:
    sub = gpu_c[gpu_c['dominant_office'] == office]
    short = office_short.get(office, office)
    ax.scatter(sub['office_entropy'], sub['hours_weighted_gpu_share'],
               s=180, color=color_map[office], edgecolor='white', linewidth=1.0,
               alpha=0.9, zorder=3, label=short)
    ax.scatter(sub['office_entropy'], sub['mean_repo_gpu_share'],
               s=180, facecolor='none', edgecolor=color_map[office],
               linewidth=2.0, alpha=0.9, zorder=2)
for _, r in gpu_c.nlargest(8, 'n_repos').iterrows():
    ax.annotate(
        r['community_label'],
        (r['office_entropy'], max(r['hours_weighted_gpu_share'], r['mean_repo_gpu_share'])),
        xytext=(8, 8), textcoords='offset points', fontsize=10, color='#333333'
    )

ax.set_xlabel('office-mix entropy within community (0 = single office)')
ax.set_ylabel('GPU share')
ax.set_title(
    f'{DISPLAY_YEAR} communities: GPU intensity vs office mixing\n'
    'filled dot = hours-weighted, hollow ring = mean per project'
)
ax.legend(title='dominant office', loc='lower right', frameon=True,
          framealpha=1.0, ncol=2, borderpad=0.7)
sns.despine(ax=ax)
plt.show()


## 2. Key Bridges

This block identifies persistent bridge projects, measures how much cross-office connectivity they carry, and compares bridge removal against a degree-matched random-removal benchmark.

In [ ]:
bridge_year_rows = []
for y in BRIDGE_ANALYSIS_YEARS:
    G = graph_by_year[y]
    office_map = office_map_by_year[y]
    bridge_year = bridge_year_counts_from_graph(
        G,
        office_map,
        seed=NULL_SEED + y,
        k=BRIDGE_BETWEENNESS_K_OBSERVED,
    )
    bridge_year['year'] = y
    bridge_year_rows.append(bridge_year)
bridge_year_df = pd.concat(bridge_year_rows, ignore_index=True)

bridge_summary = (
    bridge_year_df.groupby('repo')
    .agg(
        years_as_bridge=('is_bridge_year', 'sum'),
        mean_betweenness=('betweenness', 'mean'),
        peak_betweenness=('betweenness', 'max'),
        max_neighbor_offices=('neighbor_offices', 'max'),
    )
    .reset_index()
)

bridge_context_rows = []
for repo in bridge_summary['repo']:
    yrs = bridge_year_df[
        (bridge_year_df['repo'] == repo) &
        (bridge_year_df['is_bridge_year'])
    ]['year'].tolist()
    touched_offices, touched_science = set(), set()
    for y in yrs:
        G = graph_by_year[y]
        if not G.has_node(repo):
            continue
        for nbr in G.neighbors(repo):
            o = office_map_by_year[y].get(nbr)
            s = science_map_by_year[y].get(nbr)
            if o:
                touched_offices.add(o)
            if s:
                touched_science.add(s)
    bridge_context_rows.append({
        'repo': repo,
        'offices_spanned': len(touched_offices),
        'science_categories_spanned': len(touched_science),
    })

bridge_summary = bridge_summary.merge(pd.DataFrame(bridge_context_rows), on='repo', how='left')
persistent_bridges = (
    bridge_summary[
        (bridge_summary['years_as_bridge'] >= BRIDGE_MIN_YEARS) &
        (bridge_summary['offices_spanned'] >= BRIDGE_MIN_OFFICES)
    ]
    .sort_values(
        ['years_as_bridge', 'offices_spanned', 'peak_betweenness'],
        ascending=[False, False, False],
    )
)

brokerage_rows = []
for y in BRIDGE_ANALYSIS_YEARS:
    G = graph_by_year[y]
    office_map = office_map_by_year[y]
    year_bridges = set(
        bridge_year_df[
            (bridge_year_df['year'] == y) &
            (bridge_year_df['is_bridge_year'])
        ]['repo']
    )
    total_cross = 0
    bridge_cross = 0
    for u, v in G.edges():
        ou = office_map.get(u)
        ov = office_map.get(v)
        if ou and ov and ou != ov:
            total_cross += 1
            if u in year_bridges or v in year_bridges:
                bridge_cross += 1
    brokerage_rows.append({
        'year': y,
        'cross_office_edges_total': total_cross,
        'cross_office_edges_via_bridge': bridge_cross,
        'bridge_brokerage_share': bridge_cross / total_cross if total_cross else np.nan,
    })
brokerage_df = pd.DataFrame(brokerage_rows)

def removal_metrics(G, office_map):
    if G.number_of_nodes() == 0:
        return {'n_nodes': 0, 'n_edges': 0, 'cross_office_edges': 0, 'cross_office_share': np.nan}
    cross_edges = cross_office_edge_count(G, office_map)
    total_edges = G.number_of_edges()
    return {
        'n_nodes': G.number_of_nodes(),
        'n_edges': total_edges,
        'cross_office_edges': cross_edges,
        'cross_office_share': cross_edges / total_edges if total_edges else np.nan,
    }

removal_rows = []
random_spread_rows = []
for target_year in BRIDGE_ANALYSIS_YEARS:
    G_target = graph_by_year[target_year].copy()
    office_target = office_map_by_year[target_year]
    features_target = repo_features[repo_features['year'] == target_year][['repo', 'degree']]
    degree_map = features_target.set_index('repo')['degree'].to_dict()
    bridge_set = [r for r in persistent_bridges['repo'].tolist() if G_target.has_node(r)]

    baseline_metrics = removal_metrics(G_target, office_target)
    br_removed = G_target.copy()
    br_removed.remove_nodes_from(bridge_set)
    bridge_removed_metrics = removal_metrics(br_removed, office_target)

    deg_series = pd.Series(degree_map)
    if len(deg_series) > 1:
        deg_bins = pd.qcut(
            deg_series.rank(method='first'),
            q=min(10, len(deg_series)),
            duplicates='drop',
        )
        bin_map = deg_bins.astype(str).to_dict()
    else:
        bin_map = {r: '0' for r in deg_series.index}
    bridge_bin_counts = pd.Series([bin_map.get(r) for r in bridge_set]).value_counts()

    rng = np.random.default_rng(NULL_SEED + target_year)
    candidate_repos = [r for r in G_target.nodes() if r not in bridge_set]
    random_rows = []
    for trial in range(RANDOM_REMOVAL_TRIALS):
        chosen = []
        available = set(candidate_repos)
        for bin_name, needed in bridge_bin_counts.items():
            pool = [r for r in available if bin_map.get(r) == bin_name]
            if len(pool) >= needed:
                pick = rng.choice(pool, size=needed, replace=False).tolist()
            else:
                pick = pool
            chosen.extend(pick)
            available -= set(pick)
        if len(chosen) < len(bridge_set):
            leftovers = list(available)
            extra = rng.choice(leftovers, size=len(bridge_set) - len(chosen), replace=False).tolist()
            chosen.extend(extra)
        H = G_target.copy()
        H.remove_nodes_from(chosen)
        random_rows.append(removal_metrics(H, office_target))

    random_removal_df = pd.DataFrame(random_rows)
    rr = random_removal_df.mean(numeric_only=True).to_dict()
    rr_std = random_removal_df.std(numeric_only=True, ddof=1).to_dict()

    removal_rows.append({'year': target_year, 'scenario': 'baseline', **baseline_metrics})
    removal_rows.append({'year': target_year, 'scenario': 'persistent bridges removed', **bridge_removed_metrics})
    removal_rows.append({
        'year': target_year,
        'scenario': 'matched random mean',
        **rr,
        'cross_office_edges_std': rr_std.get('cross_office_edges', 0.0),
    })

removal_summary = pd.DataFrame(removal_rows)
plot_removal = removal_summary.copy()
scenario_order = ['baseline', 'matched random mean', 'persistent bridges removed']
plot_removal['scenario'] = pd.Categorical(
    plot_removal['scenario'],
    categories=scenario_order,
    ordered=True,
)
plot_removal = plot_removal.sort_values(['year', 'scenario'])

diag_rows = []
for y in BRIDGE_ANALYSIS_YEARS:
    base = removal_summary[
        (removal_summary['year'] == y) &
        (removal_summary['scenario'] == 'baseline')
    ].iloc[0]
    for scenario in ['matched random mean', 'persistent bridges removed']:
        sub = removal_summary[
            (removal_summary['year'] == y) &
            (removal_summary['scenario'] == scenario)
        ].iloc[0]
        diag_rows.append({
            'year': y,
            'scenario': scenario,
            'cross_office_edge_retention': sub['cross_office_edges'] / base['cross_office_edges'],
            'total_edge_retention': sub['n_edges'] / base['n_edges'],
        })
diagnostics = pd.DataFrame(diag_rows)
diag_retention = diagnostics.melt(
    id_vars=['year', 'scenario'],
    value_vars=['cross_office_edge_retention', 'total_edge_retention'],
    var_name='edge_type',
    value_name='retention',
)
diag_retention['series'] = diag_retention['scenario'] + ': ' + diag_retention['edge_type'].map({
    'cross_office_edge_retention': 'cross-office',
    'total_edge_retention': 'all edges',
})

print(f'persistent bridges identified: {len(persistent_bridges)}')


### 2.1 Cross-office brokerage carried by bridge projects

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=brokerage_df, x='year', y='bridge_brokerage_share',
             marker='o', linewidth=3.0, markersize=10, color=PALETTE[0], ax=ax)
ax.fill_between(brokerage_df['year'], 0, brokerage_df['bridge_brokerage_share'],
                color=PALETTE[0], alpha=0.14)
ax.set_title('Cross-office brokerage carried by the bridge layer')
ax.set_ylabel('share via bridge projects')
ax.set_xlabel('year')
ax.set_xticks(FULL_YEARS)
ax.set_ylim(0, 1)
for _, r in brokerage_df.iterrows():
    ax.annotate(f"{r['bridge_brokerage_share']:.2f}",
                (r['year'], r['bridge_brokerage_share']),
                textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=11, color=PALETTE[0], fontweight='bold')
sns.despine(ax=ax)
plt.show()


### 2.2 Top persistent bridge projects

In [ ]:
persistent_bridges[
    ['repo', 'years_as_bridge', 'offices_spanned', 'science_categories_spanned',
     'mean_betweenness', 'peak_betweenness']
].head(20)


### 2.3 Sensitivity to the persistent-bridge threshold

In [ ]:
threshold_rows = []
for pct in [0.85, 0.90, 0.95]:
    for min_years in [4, 5, 6]:
        tmp = []
        for y in BRIDGE_ANALYSIS_YEARS:
            sub = bridge_year_df[bridge_year_df['year'] == y][['repo', 'betweenness', 'neighbor_offices']].copy()
            thresh = sub['betweenness'].quantile(pct)
            sub['is_bridge_year_alt'] = (
                (sub['betweenness'] >= thresh) &
                (sub['neighbor_offices'] >= BRIDGE_MIN_OFFICES)
            )
            sub['year'] = y
            tmp.append(sub[['repo', 'year', 'is_bridge_year_alt']])
        tmp = pd.concat(tmp, ignore_index=True)
        n_persistent = (tmp.groupby('repo')['is_bridge_year_alt'].sum() >= min_years).sum()
        threshold_rows.append({
            'top_percentile': int(pct * 100),
            'min_years': min_years,
            'n_persistent_repos': int(n_persistent),
        })

threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_pivot = threshold_sensitivity.pivot(
    index='top_percentile',
    columns='min_years',
    values='n_persistent_repos',
)

fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(threshold_pivot, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': '# persistent bridge projects'}, ax=ax)
ax.set_xlabel('minimum bridge years')
ax.set_ylabel('top betweenness percentile')
ax.set_title('Persistent bridge count under alternative thresholds')
plt.show()

threshold_pivot


### 2.4 Cross-office edges under baseline, matched-random, and bridge-removal scenarios

In [ ]:
palette_map = {
    'baseline': '#2F6DA3',
    'matched random mean': '#F28E2B',
    'persistent bridges removed': '#D1495B',
}
years = sorted(plot_removal['year'].unique())

fig, ax = plt.subplots(figsize=(11, 6.5))
sns.barplot(data=plot_removal, x='year', y='cross_office_edges', hue='scenario',
            hue_order=scenario_order, palette=palette_map, ax=ax, width=0.7)

sub_random = (
    plot_removal[plot_removal['scenario'] == 'matched random mean']
    .set_index('year').loc[years]
)
random_bars = ax.patches[len(years):len(years) * 2]
for patch, (_, row) in zip(random_bars, sub_random.iterrows()):
    ax.errorbar(
        patch.get_x() + patch.get_width() / 2,
        row['cross_office_edges'],
        yerr=row.get('cross_office_edges_std', 0.0),
        fmt='none',
        ecolor='black',
        elinewidth=1.3,
        capsize=5,
        alpha=0.85,
    )

ax.set_title('Cross-office edges by scenario')
ax.set_xlabel('year')
ax.set_ylabel('cross-office edges')
ax.grid(axis='y', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
vals = plot_removal['cross_office_edges'].to_numpy()
pad = (vals.max() - vals.min()) * 0.15
ax.set_ylim(max(0, vals.min() - pad), vals.max() + pad)
ax.legend(frameon=False, loc='upper left', title='scenario')
plt.show()


### 2.5 Bridge removal hits cross-office edges harder than total edge count

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))
sns.lineplot(data=diag_retention, x='year', y='retention',
             hue='series', style='series', markers=True, dashes=True,
             linewidth=2.8, markersize=10, ax=ax)
ax.set_title('Fraction of the baseline network retained after removal')
ax.set_ylabel('fraction of baseline retained')
ax.set_xlabel('year')
ax.set_xticks(sorted(diag_retention['year'].unique()))
ax.set_ylim(0.55, 1.02)
ax.legend(frameon=False, loc='upper left', ncol=2, title='')
sns.despine(ax=ax)
plt.show()


## 3. Liaison Discovery

This block builds a user-level liaison score, visualizes the user-project network structure behind it, and evaluates how many users the top-ranked candidates can reach through shared projects.

In [ ]:

users_meta = pd.read_csv('../clean/users.csv')

edges_y = (
    alloc[alloc['year'] == LIAISON_YEAR][['repo', 'user_id']]
    .dropna()
    .drop_duplicates()
)
breadth_y = edges_y.groupby('user_id')['repo'].nunique().rename('breadth_y')
repo_size_y = edges_y.groupby('repo')['user_id'].nunique().rename('repo_size_y')
max_proj = (
    edges_y.merge(repo_size_y, on='repo')
    .groupby('user_id')['repo_size_y'].max()
    .rename('max_project_size')
)

B = nx.Graph()
B.add_nodes_from(('u::' + edges_y['user_id']).unique(), bipartite='user')
B.add_nodes_from(('r::' + edges_y['repo']).unique(), bipartite='repo')
B.add_edges_from(zip('u::' + edges_y['user_id'], 'r::' + edges_y['repo']))

btw = nx.betweenness_centrality(B, k=min(500, B.number_of_nodes()), seed=42)
user_btw = pd.Series(
    {n[3:]: v for n, v in btw.items() if n.startswith('u::')},
    name='bipartite_betweenness',
)

liaison_features = (
    users_meta[['user_id', 'tenure', 'n_repos_lifetime']]
    .rename(columns={'n_repos_lifetime': 'breadth_lifetime'})
    .merge(breadth_y, left_on='user_id', right_index=True, how='left')
    .merge(max_proj, left_on='user_id', right_index=True, how='left')
    .merge(user_btw, left_on='user_id', right_index=True, how='left')
)
liaison_features = liaison_features.dropna(subset=['breadth_y']).copy()
liaison_features[['max_project_size', 'bipartite_betweenness']] = (
    liaison_features[['max_project_size', 'bipartite_betweenness']].fillna(0)
)
score_cols = ['tenure', 'breadth_y', 'max_project_size', 'bipartite_betweenness']
score_mean = liaison_features[score_cols].mean()
score_std = liaison_features[score_cols].std().replace(0, 1)
z = (liaison_features[score_cols] - score_mean) / score_std
liaison_features['liaison_score'] = z.mean(axis=1)
liaison_features = liaison_features.sort_values('liaison_score', ascending=False).reset_index(drop=True)
liaison_features['rank'] = liaison_features.index + 1
anchor_liaisons = liaison_features.head(4)['user_id'].tolist()

def reach(uids, edges):
    repos = set(edges[edges['user_id'].isin(uids)]['repo'])
    return set(edges[edges['repo'].isin(repos)]['user_id'])

total_users = edges_y['user_id'].nunique()
candidates_sorted = liaison_features['user_id'].tolist()
candidate_count = len(candidates_sorted)
N_GRID = [n for n in [5, 10, 25, 50, 100, 200, 500, 1000] if n <= candidate_count]
if candidate_count and (not N_GRID or N_GRID[-1] < candidate_count):
    N_GRID.append(candidate_count)

rng = np.random.default_rng(42)
breadth_weight = (
    liaison_features.set_index('user_id')['breadth_y']
    .reindex(candidates_sorted).fillna(0).to_numpy() + 0.001
)
breadth_weight = breadth_weight / breadth_weight.sum()

rows = []
for n in N_GRID:
    top_n = candidates_sorted[:n]
    cov_top = len(reach(top_n, edges_y)) / total_users
    cov_rand = []
    for _ in range(100):
        rand_n = rng.choice(candidates_sorted, size=n, replace=False, p=breadth_weight)
        cov_rand.append(len(reach(rand_n, edges_y)) / total_users)
    rows.append({
        'N': n,
        'coverage_top': cov_top,
        'coverage_rand_mean': float(np.mean(cov_rand)),
        'coverage_rand_p95': float(np.percentile(cov_rand, 95)),
    })
coverage = pd.DataFrame(rows)

print(f'users scored for liaison ranking: {len(liaison_features):,}')


### 3.1 User-project network snapshot

In [ ]:

repo_nodes_full = [n for n, d in B.nodes(data=True) if d.get('bipartite') == 'repo']
repo_degrees = pd.Series({n: B.degree(n) for n in repo_nodes_full})
MIN_USERS_PER_REPO = max(2, int(repo_degrees.quantile(0.60))) if not repo_degrees.empty else 1
MIN_REPOS_PER_USER = 2

kept_repos = {r for r in repo_nodes_full if B.degree(r) >= MIN_USERS_PER_REPO}
user_hits = {}
for r in kept_repos:
    for u in B.neighbors(r):
        user_hits[u] = user_hits.get(u, 0) + 1
kept_users = {u for u, k in user_hits.items() if k >= MIN_REPOS_PER_USER}

sub = B.subgraph(kept_repos | kept_users).copy()
if sub.number_of_nodes() == 0:
    sub = B.copy()
if sub.number_of_nodes() > 0:
    sub = sub.subgraph(max(nx.connected_components(sub), key=len)).copy()

sub_repos = [n for n in sub.nodes if n.startswith('r::')]
sub_users = [n for n in sub.nodes if n.startswith('u::')]
n_repos, n_users = len(sub_repos), len(sub_users)
pos = nx.spring_layout(sub, seed=42, k=1.2 / (sub.number_of_nodes() ** 0.5), iterations=80)

node_df = pd.DataFrame({
    'x': [pos[n][0] for n in sub.nodes],
    'y': [pos[n][1] for n in sub.nodes],
    'kind': ['repo' if n.startswith('r::') else 'user' for n in sub.nodes],
    'degree': [sub.degree(n) for n in sub.nodes],
})

fig, ax = plt.subplots(figsize=(11, 8))
ax.add_collection(LineCollection([(pos[u], pos[v]) for u, v in sub.edges],
                                 colors=PALETTE[4], alpha=0.18, linewidths=0.7))
sns.scatterplot(data=node_df[node_df['kind'] == 'user'], x='x', y='y',
                color=PALETTE[0], s=35, alpha=0.75, linewidth=0, ax=ax, legend=False)
sns.scatterplot(data=node_df[node_df['kind'] == 'repo'], x='x', y='y',
                size='degree', sizes=(110, 650), color=PALETTE[1], alpha=0.9,
                edgecolor='white', linewidth=1.0, ax=ax, legend=False)
ax.legend(handles=[
    Line2D([0], [0], marker='o', color='none', markerfacecolor=PALETTE[0],
           markersize=10, markeredgewidth=0, label=f'users (n={n_users:,})'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=PALETTE[1],
           markersize=14, markeredgecolor='white', markeredgewidth=1.0,
           label=f'projects (n={n_repos})'),
], loc='upper right', frameon=True, handletextpad=0.6)
ax.set_axis_off()
ax.set_title(
    f'User-project network snapshot for {LIAISON_YEAR}\n'
    f'(projects with at least {MIN_USERS_PER_REPO} users, users on at least {MIN_REPOS_PER_USER} of them)'
)
plt.show()


### 3.2 Coverage from contacting the top-ranked liaison candidates

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(coverage['N'], coverage['coverage_top'] * 100,
        marker='o', linewidth=3.0, markersize=10, color=PALETTE[2],
        label='top-N predicted liaisons')
ax.fill_between(coverage['N'],
                coverage['coverage_rand_mean'] * 100,
                coverage['coverage_rand_p95'] * 100,
                color=PALETTE[0], alpha=0.18,
                label='breadth-matched random (mean to p95, n=100)')
ax.plot(coverage['N'], coverage['coverage_rand_mean'] * 100,
        linestyle='--', color=PALETTE[0], linewidth=2.1, alpha=0.9)
ax.set_xscale('log')
ax.set_xlabel('N candidates contacted')
ax.set_ylabel('% users reached')
ax.set_title(f'Coverage of {LIAISON_YEAR} users by contacting top-N candidates')
ax.set_ylim(0, 115)
ax.legend(loc='lower right', frameon=False)
ax.set_xticks(list(coverage['N']))
ax.set_xticklabels([str(n) for n in coverage['N']])
ax.tick_params(axis='x', which='minor', length=0)
ax.get_xaxis().set_minor_formatter(plt.NullFormatter())
for n, cov in zip(coverage['N'], coverage['coverage_top']):
    ax.annotate(f"{cov * 100:.0f}%", (n, cov * 100),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=11, color=PALETTE[2], fontweight='bold')
plt.show()


### 3.3 Liaison personas in feature space

In [ ]:
scatter_df = liaison_features[
    (liaison_features['breadth_y'] > 0) &
    (liaison_features['max_project_size'] > 0)
].copy()

fig, ax = plt.subplots(figsize=(11, 7.5))
COLOR_MAX = 1.5
sc = ax.scatter(scatter_df['breadth_y'], scatter_df['max_project_size'],
                c=scatter_df['liaison_score'].clip(upper=COLOR_MAX),
                cmap='viridis', vmin=0, vmax=COLOR_MAX,
                s=22, alpha=0.55, edgecolor='none')
cb = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
cb.set_label(f'liaison_score (clipped at {COLOR_MAX})')

ax.set_xscale('log')
ax.set_yscale('log')
x_ticks = [1, 2, 3, 5, 7, 10, 15, 20]
x_ticks = [t for t in x_ticks if t <= scatter_df['breadth_y'].max() * 1.05]
ax.xaxis.set_major_locator(FixedLocator(x_ticks))
ax.xaxis.set_major_formatter(FixedFormatter([str(t) for t in x_ticks]))
ax.xaxis.set_minor_formatter(NullFormatter())
y_ticks = [1, 3, 10, 30, 100, 300, 1000]
y_ticks = [t for t in y_ticks if t <= scatter_df['max_project_size'].max() * 1.05]
ax.yaxis.set_major_locator(FixedLocator(y_ticks))
ax.yaxis.set_major_formatter(FixedFormatter([str(t) for t in y_ticks]))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.tick_params(axis='both', which='minor', length=0)

ax.set_xlabel(f'breadth: # projects in {LIAISON_YEAR} (log)')
ax.set_ylabel('max project size in users (log)')
ax.set_title(
    f'Two liaison personas in feature space — {LIAISON_YEAR}\n'
    'x = # projects this user is on • y = biggest project they are on, in users'
)

named = scatter_df[scatter_df['user_id'].isin(anchor_liaisons)].copy()
ax.scatter(named['breadth_y'], named['max_project_size'], s=320,
           edgecolor=PALETTE[2], facecolor='none', linewidth=3, zorder=4)
anchor_handle = Line2D([0], [0], marker='o', linestyle='None',
                       label='anchor user',
                       markerfacecolor='none', markeredgecolor=PALETTE[2],
                       markeredgewidth=3, markersize=12)
ax.legend(handles=[anchor_handle], loc='lower right', frameon=True)
sns.despine(ax=ax)
plt.show()


### 3.4 Top-ranked liaison candidates

In [ ]:
liaison_features[
    ['rank', 'user_id', 'liaison_score', 'tenure', 'breadth_y',
     'max_project_size', 'bipartite_betweenness', 'breadth_lifetime']
].head(20)
